# Análise de vendas

Análise da base do desafio usando pandas e matplotlib.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
vendas = pd.read_excel("../Base-Dados-Desafio-500k.xlsx", sheet_name="VENDAS")
produtos = pd.read_excel("../Base-Dados-Desafio-500k.xlsx", sheet_name="PRODUTOS")


In [ ]:
vendas.head()


In [ ]:
produtos


## olhando a base


In [ ]:
vendas.shape


In [ ]:
vendas.info()


In [ ]:
vendas.isnull().sum()


In [ ]:
vendas.duplicated().sum()


In [ ]:
vendas.describe()


## tratamento


In [ ]:
# removendo linhas repetidas
vendas = vendas.drop_duplicates()


In [ ]:
# ajustando os tipos
vendas["DATA"] = pd.to_datetime(vendas["DATA"], errors="coerce")
vendas["CLIENTE"] = pd.to_numeric(vendas["CLIENTE"], errors="coerce")
vendas["IDADE"] = pd.to_numeric(vendas["IDADE"], errors="coerce")
vendas["QUANTIDADE_VENDIDA"] = pd.to_numeric(vendas["QUANTIDADE_VENDIDA"], errors="coerce")
vendas["PREÇO_UNITARIO"] = pd.to_numeric(vendas["PREÇO_UNITARIO"], errors="coerce")


In [ ]:
# deixando os textos no mesmo padrão
vendas["ESTADO"] = vendas["ESTADO"].str.strip().str.upper()
vendas["PRODUTO"] = vendas["PRODUTO"].str.strip().str.upper()

produtos["PRODUTO"] = produtos["PRODUTO"].str.strip().str.upper()


In [ ]:
# trazendo a categoria da outra aba
vendas = vendas.merge(produtos, on="PRODUTO", how="left")

vendas.head()


In [ ]:
vendas.isnull().sum()


In [ ]:
# removendo dados que não ajudam nas análises
vendas = vendas.dropna(subset=["CLIENTE", "IDADE", "ESTADO", "PRODUTO", "CATEGORIA", "DATA"])

vendas = vendas[vendas["IDADE"] >= 18]
vendas = vendas[vendas["IDADE"] <= 80]

vendas = vendas[vendas["QUANTIDADE_VENDIDA"] > 0]
vendas = vendas[vendas["PREÇO_UNITARIO"] > 0]

vendas.shape


In [ ]:
# valor total da venda
vendas["VALOR_TOTAL"] = vendas["QUANTIDADE_VENDIDA"] * vendas["PREÇO_UNITARIO"]

vendas.head()


## perfil dos clientes


In [ ]:
vendas["IDADE"].mean()


In [ ]:
vendas["IDADE"].median()


In [ ]:
# separando por faixa de idade
vendas["FAIXA_ETARIA"] = pd.cut(
    vendas["IDADE"],
    bins=[17, 24, 34, 44, 54, 64, 100],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
)

vendas["FAIXA_ETARIA"].value_counts().sort_index()


In [ ]:
vendas["FAIXA_ETARIA"].value_counts().sort_index().plot(kind="bar")
plt.title("Clientes por faixa etária")
plt.xlabel("Faixa etária")
plt.ylabel("Quantidade")
plt.show()


In [ ]:
vendas["ESTADO"].value_counts()


A faixa de 35 a 44 anos aparece bastante na base.
Também é possível ver que São Paulo possui mais registros que os outros estados.


## categorias de produtos


In [ ]:
# faturamento por categoria
faturamento_categoria = vendas.groupby("CATEGORIA")["VALOR_TOTAL"].sum().sort_values(ascending=False)
faturamento_categoria


In [ ]:
# quantidade vendida
volume_categoria = vendas.groupby("CATEGORIA")["QUANTIDADE_VENDIDA"].sum().sort_values(ascending=False)
volume_categoria


In [ ]:
# ticket médio simples
ticket_categoria = vendas.groupby("CATEGORIA")["VALOR_TOTAL"].mean().sort_values(ascending=False)
ticket_categoria


In [ ]:
faturamento_categoria.plot(kind="bar")
plt.title("Faturamento por categoria")
plt.ylabel("Faturamento")
plt.show()


Pelo faturamento, algumas categorias ficam bem próximas.
Jardinagem aparece entre as categorias de maior faturamento e Mangueiras tem bastante volume vendido.


## vendas ao longo do tempo


In [ ]:
# pegando mês e ano
vendas["MES"] = vendas["DATA"].dt.month
vendas["ANO"] = vendas["DATA"].dt.year


In [ ]:
faturamento_mes = vendas.groupby("MES")["VALOR_TOTAL"].sum()
faturamento_mes


In [ ]:
faturamento_mes.plot(kind="bar")
plt.title("Faturamento por mês")
plt.xlabel("Mês")
plt.ylabel("Faturamento")
plt.show()


In [ ]:
vendas.groupby("ANO")["VALOR_TOTAL"].sum()


Os meses possuem algumas diferenças, mas no geral não parece existir uma sazonalidade muito forte só olhando esse gráfico.


## análise por estado


In [ ]:
faturamento_estado = vendas.groupby("ESTADO")["VALOR_TOTAL"].sum().sort_values(ascending=False)
faturamento_estado


In [ ]:
volume_estado = vendas.groupby("ESTADO")["QUANTIDADE_VENDIDA"].sum().sort_values(ascending=False)
volume_estado


In [ ]:
vendas.groupby(["ANO", "ESTADO"])["VALOR_TOTAL"].sum().unstack().plot()
plt.title("Faturamento por estado ao longo dos anos")
plt.ylabel("Faturamento")
plt.show()


São Paulo apresenta o maior faturamento.
No gráfico por ano, os estados seguem um comportamento relativamente parecido.


## idade e categoria


In [ ]:
idade_categoria = vendas.groupby(
    ["FAIXA_ETARIA", "CATEGORIA"]
)["QUANTIDADE_VENDIDA"].sum()

idade_categoria


In [ ]:
tabela_idade_categoria = idade_categoria.unstack()
tabela_idade_categoria


In [ ]:
tabela_idade_categoria.plot(kind="bar")
plt.title("Categorias por faixa etária")
plt.ylabel("Quantidade vendida")
plt.show()


In [ ]:
tabela_idade_categoria.idxmax(axis=1)


## salvando a base tratada


In [ ]:
vendas.to_csv("../vendas_tratadas.csv", index=False, sep=";")
